# Feature preprocessing pipeline
## Main goal

This notebook builds the schema driven processing pipeline that turns the features created in the extraction pipeline into a clean, fully numeric matrix, ready for model training.

The schema used was defined as the variables were constructed (see the Features Extraction Notebook), or by the variable distribution in the application data (s. The schema consists of the following info:
* type
* expected range
* imputation value
    
## Preproccess data pipeline
    This process assembles a 'scikit-learn' / 'feature-engine' 'Pipeline' that:
1. **Selects** the exact set of features defined in the schema.
2. **Converts data types**, casting numerical features to `float64` and cleaning up
   categorical values into a consistent string format.
3. **Imputes missing values** — categorical variables are filled with an explicit
   `"ND"` placeholder, while numerical variables are filled per the imputation value
   defined for each feature in the schema.
4. **Caps outliers** using the min/max bounds declared in the schema.
5. **Encodes categorical variables**, grouping rare categories together and applying
   target-guided ordinal encoding.
6. **Scales numerical features** (`StandardScaler` by default).

The resulting pipeline is fit once on the training data, serialized with `joblib` for
reuse at inference time, and applied to both the train and test sets to produce the
final preprocessed CSV files used downstream for model training.

> **Note:** the preprocessor is entirely schema-driven — adding, removing, or
> reclassifying a feature only requires updating `data_schema.json`, with no changes
> needed to the pipeline code itself.

In [1]:
import numpy as np
import pandas as pd
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
from collections import OrderedDict
import json
import joblib

#
from pathlib import Path

#Pipeline elements
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from feature_engine.imputation import CategoricalImputer
from feature_engine.encoding import RareLabelEncoder, OrdinalEncoder
from feature_engine.wrappers import SklearnTransformerWrapper
from feature_engine.outliers import ArbitraryOutlierCapper
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from functools import partial


pd.set_option('display.max_columns', None)


In [2]:
# Project root: this notebook lives in <ROOT>/notebooks. Safe to re-run.
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT_DIR)
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

# Everything below lives in src/ (importing it, instead of redefining it here,
# keeps the saved preprocessor loadable from other notebooks and scripts)
from src.model_paths import (
    DATA_DIR, SCHEMA_DIR,
    WritePathsPreprocess, ReadPathsPreprocess,
)
from src.utils import load_data_schema, get_feature_lists, recast_numeric
from src.transform import ManualSelector, TypeConverter
from src.preprocessor_utils import (
    create_initial_pipeline, get_preprocessor, save_preprocessed_data, get_column_names,
)

In [3]:
#Load Data And Schema
data_train_path = DATA_DIR / "application_train_all_features.csv"
data_test_path = DATA_DIR / "application_test_all_features.csv"
data_schema_path = SCHEMA_DIR/"data_schema.json"
model_name = "model_nb"  # light model on a small sample, for faster execution
SAMPLE_N, SAMPLE_SEED = 5000, 42  # rows sampled from each file, and the seed

In [4]:
#Load Data and Schema

target_col = "TARGET"
other_metrics = ["SK_ID_CURR"] #ID columns and columns not related with model

In [5]:
path_preprocessor = WritePathsPreprocess(model_name)


In [6]:
schema = load_data_schema(data_schema_path)
train_final = pd.read_csv(data_train_path).sample(SAMPLE_N, random_state=SAMPLE_SEED)
# Test rows must not appear in train: draw them from the rows the train sample did not take
test_pool = pd.read_csv(data_test_path)
test_pool = test_pool[~test_pool["SK_ID_CURR"].isin(train_final["SK_ID_CURR"])]
test_final = test_pool.sample(min(SAMPLE_N, len(test_pool)), random_state=SAMPLE_SEED)
print(f"train {train_final.shape}, test {test_final.shape}, shared ids: "
      f"{len(set(train_final['SK_ID_CURR']) & set(test_final['SK_ID_CURR']))}")
X = train_final
y = train_final[target_col]
#Start preprocessor
path_preprocessor = WritePathsPreprocess(model_name)
preprocessor, column_names = get_preprocessor(
     x = X,
     y = y,
     target_col = target_col,
     other_metrics = other_metrics,
     data_schema_path = data_schema_path,
   
     write_paths = path_preprocessor
 )
#Preprocesed data
save_preprocessed_data(
        x = X,
        y = y,
        preprocessor = preprocessor,
        column_names = column_names,
        other_metrics = other_metrics,
        target = target_col,
        path = path_preprocessor.train_dataset_path,
        save = True
)
save_preprocessed_data(
        x = test_final,
        y = test_final["TARGET"],
        preprocessor = preprocessor,
        column_names = column_names,
        other_metrics = other_metrics,
        target = target_col,
        path = path_preprocessor.test_dataset_path,
        save = True
)
model_columns = get_column_names(preprocessor)
#preprocessor[-1].get_feature_names_out()

train (5000, 352), test (5000, 352), shared ids: 0
[Pipeline] .. (step 1 of 10) Processing manual_selector, total=   0.0s
[Pipeline] ... (step 2 of 10) Processing type_converter, total=   0.3s
[Pipeline]  (step 3 of 10) Processing categorical_imputer, total=   0.0s
[Pipeline] . (step 4 of 10) Processing simple_imputer_0, total=   0.2s
[Pipeline]  (step 5 of 10) Processing simple_imputer_ND, total=   0.1s
[Pipeline] ... (step 6 of 10) Processing numeric_recast, total=   0.0s
[Pipeline] ... (step 7 of 10) Processing outlier_capper, total=   0.1s
[Pipeline]  (step 8 of 10) Processing rare_label_encoder, total=   0.1s


/Users/oyieth/Documents/Python projects/credit_scoring_examples/venv/lib/python3.14/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable freq_contract_type is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/Users/oyieth/Documents/Python projects/credit_scoring_examples/venv/lib/python3.14/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable freq_status is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/Users/oyieth/Documents/Python projects/credit_scoring_examples/venv/lib/python3.14/site-packages/feature_engine/encoding/rare_label.py:216: UserWarning: The number of unique categories for variable freq_payment_type is less than that indicated in n_categories. Thus, all categories will be considered frequent
  warnings.warn(
/Users/oyieth/Documen

[Pipeline] .. (step 9 of 10) Processing ordinal_encoder, total=   0.1s
[Pipeline] .. (step 10 of 10) Processing simple_imputer, total=   0.0s
x_preprocessed inital pipeline: Done
[Pipeline] ............ (step 1 of 1) Processing scaler, total=   0.0s
Preprocessor saved at /Users/oyieth/Documents/Python projects/credit_scoring_examples/models/model_nb/artifacts/preprocessor_model_nb.joblib.dat & Ordinal encoder dict saved at /Users/oyieth/Documents/Python projects/credit_scoring_examples/models/model_nb/schema/ordinal_encoder_dictmodel_nb.json!
Dataset saved at /Users/oyieth/Documents/Python projects/credit_scoring_examples/models/model_nb/data/train_model_nb_preprocessed.csv!
Dataset saved at /Users/oyieth/Documents/Python projects/credit_scoring_examples/models/model_nb/data/test_model_nb_preprocessed.csv!
